# Rheology Analysis Dashboard

End-to-end cone–plate viscometry analysis powered by `rheology_pipeline_core`.

This notebook ingests **dynamic analysis** CSV exports (`dynamic_analysis_*.csv`):
comment-prefixed metadata followed by long-format rows with `Z_Height_mm`, `RPM`,
and `Torque_%`. Each RPM sweep is extracted automatically; multi-RPM runs produce
one drag profile per spindle speed. Silicone calibration, flow-regime classification,
and diagnostic plots follow.

## 1. Setup and Imports

In [ ]:
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

try:
    import seaborn as sns
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn>=0.13", "-q"])
    import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

# Ensure this notebook's directory is on sys.path for rheology_pipeline_core
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "rheology_pipeline_core.py").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "rheology_pipeline_core.py").exists():
            NOTEBOOK_DIR = candidate
            break
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from rheology_pipeline_core import create_default_pipeline, RheologyPipeline

# Publication-style matplotlib defaults
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (11, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--",
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.constrained_layout.use": True,
})
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)

## 2. User Configuration & Data Loading

Set `DATA_FILE_PATH` to a `dynamic_analysis_*.csv` export. Use `CELL_ID` to select
which well/cell to analyse (multi-RPM runs list several RPM values per Z-height).
Point `CALIBRATION_FILE_PATH` at a silicone calibration file, or set it to `None`
for factory defaults.

In [ ]:
# ── User configuration ──────────────────────────────────────────────────────
DATA_FILE_PATH = "dynamic_analysis_11_61kcP_custom_20260615_085140.csv"
CALIBRATION_FILE_PATH = "../../Auto-runs/height_normalized.csv"  # or None
CELL_ID = 1          # which cell/well to analyse
CELL_LABEL = None    # optional: e.g. "Trial 1"; overrides CELL_ID when set
MIN_TORQUE_PCT = 0.0 # drop samples at or below this torque (use 25 to skip hunt skips)


def load_dynamic_analysis_csv(path: Path) -> pd.DataFrame:
    """Load a dynamic analysis export, skipping metadata preamble lines."""
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    try:
        header_idx = next(i for i, line in enumerate(lines) if line.startswith("row,"))
    except StopIteration as exc:
        raise ValueError(f"No data header found in {path.name}") from exc
    return pd.read_csv(path, skiprows=range(header_idx))


data_path = (NOTEBOOK_DIR / DATA_FILE_PATH).resolve()
cal_path = (NOTEBOOK_DIR / CALIBRATION_FILE_PATH).resolve() if CALIBRATION_FILE_PATH else None

if not data_path.exists():
    raise FileNotFoundError(f"Sample data not found: {data_path}")

raw_df = load_dynamic_analysis_csv(data_path)
print(f"Loaded {data_path.name}: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns")

required_cols = {"Z_Height_mm", "RPM", "Torque_%"}
missing = required_cols - set(raw_df.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {sorted(missing)}")

cells = sorted(raw_df["cell"].dropna().unique())
rpms = sorted(raw_df["RPM"].dropna().unique())
print(f"Cells available : {cells}")
print(f"RPMs in file    : {rpms}")
display(raw_df[["cell", "Cell_Label", "Z_Height_mm", "RPM", "Torque_%"]].head(10))

## 3. Automated Data Parsing & Processing

Rows are filtered to the selected cell, grouped by `RPM`, and converted into
per-trial height/torque arrays. Heights are re-zeroed to the minimum Z position
in each sweep; invalid or low-torque samples are dropped automatically.

In [ ]:
def parse_dynamic_analysis(df, cell_id=1, cell_label=None, min_torque_pct=0.0):
    """Extract per-RPM height/torque sweeps from a dynamic analysis export."""
    sub = df.copy()
    if cell_label:
        sub = sub[sub["Cell_Label"].astype(str) == str(cell_label)]
    else:
        sub = sub[sub["cell"] == cell_id]

    if sub.empty:
        raise ValueError(f"No rows found for cell_id={cell_id}, cell_label={cell_label!r}")

    trials = []
    for rpm, group in sub.groupby("RPM"):
        group = group.sort_values("Z_Height_mm")
        h_raw = pd.to_numeric(group["Z_Height_mm"], errors="coerce").values.astype(float)
        torque_pct = pd.to_numeric(group["Torque_%"], errors="coerce").values.astype(float)

        h_mm = h_raw - np.nanmin(h_raw)
        valid = (
            np.isfinite(h_mm)
            & np.isfinite(torque_pct)
            & (torque_pct > min_torque_pct)
        )
        h_mm, torque_pct = h_mm[valid], torque_pct[valid]

        if len(h_mm) < 4:
            continue

        trials.append({
            "rpm": float(rpm),
            "height_mm": h_mm,
            "torque_pct": torque_pct,
            "n_points": len(h_mm),
        })

    if not trials:
        raise ValueError("No valid RPM sweeps parsed from the selected cell.")

    trials.sort(key=lambda t: t["rpm"])
    return trials


trials = parse_dynamic_analysis(
    raw_df,
    cell_id=CELL_ID,
    cell_label=CELL_LABEL,
    min_torque_pct=MIN_TORQUE_PCT,
)
summary = pd.DataFrame([
    {
        "rpm": t["rpm"],
        "n_points": t["n_points"],
        "h_range_mm": f"{t['height_mm'].min():.3f}–{t['height_mm'].max():.3f}",
        "torque_range_%": f"{t['torque_pct'].min():.1f}–{t['torque_pct'].max():.1f}",
    }
    for t in trials
])
print(f"Parsed {len(trials)} valid RPM trial(s) for cell {CELL_LABEL or CELL_ID}.")
display(summary)

## 4. Calibration Engine

Initialize the pipeline and load silicone calibration when a file is provided.
Otherwise fall back to factory-default $h_c$, $k$, and $p$.

In [ ]:
if cal_path is not None and cal_path.exists():
    pipeline = RheologyPipeline()
    cal_info = pipeline.load_silicone_calibration(cal_path)
    print("Silicone calibration loaded from:", cal_path)
    print(f"  h_c = {cal_info['h_c']:.4f} mm")
    print(f"  k   = {cal_info['k']:.4e}")
    print(f"  p   = {cal_info['p']:.4f}")
    print(f"  R²  = {cal_info['R2_calibration']:.4f}  ({cal_info['n_silicones']} silicone traces)")
elif cal_path is not None:
    pipeline = create_default_pipeline()
    print(f"Calibration file not found ({cal_path}) — using factory-default parameters.")
    print(f"  h_c = {pipeline.H_C_UNIVERSAL:.4f} mm")
    print(f"  k   = {pipeline.SILICONE_K:.4e}")
    print(f"  p   = {pipeline.SILICONE_P:.4f}")
else:
    pipeline = create_default_pipeline()
    print("No calibration file provided — using factory-default parameters.")
    print(f"  h_c = {pipeline.H_C_UNIVERSAL:.4f} mm")
    print(f"  k   = {pipeline.SILICONE_K:.4e}")
    print(f"  p   = {pipeline.SILICONE_P:.4f}")

## 5. Pipeline Execution & Rheology Classification

Parsed trials are passed to `predict_rheology()`. Single-RPM data are analysed
as Newtonian; multi-RPM data receive a power-law fit and regime classification.

In [ ]:
if len(trials) == 1:
    t = trials[0]
    result = pipeline.predict_rheology(t["height_mm"], t["torque_pct"], t["rpm"])
else:
    result = pipeline.predict_rheology(
        [t["height_mm"] for t in trials],
        [t["torque_pct"] for t in trials],
        [t["rpm"] for t in trials],
    )

r2_key = "R2_powerlaw" if result.get("mode") == "powerlaw" else "fit_quality"
r2_val = result.get(r2_key, np.nan)
k_cP = result.get("K_Pas_n", np.nan) / pipeline.geo.CP_TO_PAS

print("\n" + "═" * 52)
print("           RHEOLOGY ANALYSIS SUMMARY")
print("═" * 52)
print(f"  Flow Regime Classification : {result.get('regime', 'N/A')}")
print(f"  Flow-behavior index  (n)   : {result.get('n', np.nan):.4f}")
print(f"  Consistency index    (K)   : {k_cP:.4g} cP·s^(n−1)")
print(f"  Power-law fit quality (R²) : {r2_val:.4f}" if np.isfinite(r2_val) else "  Power-law fit quality (R²) : N/A")
if result.get("mode") == "newtonian" and np.isfinite(result.get("mu_app_cP", np.nan)):
    print(f"  Apparent viscosity (μ_app) : {result['mu_app_cP']:.2f} cP")
print("═" * 52)

## 6. Dynamic Visualizations

**Left** — raw drag profiles $D = T/\mathrm{RPM}$ vs. re-zeroed height with
calibrated fractional fits $A/(h + h_c) + B$.

**Right** — log–log rheology curve: apparent viscosity $\mu_\mathrm{app}$ (cP)
vs. shear rate $\dot{\gamma}$ ($\mathrm{s}^{-1}$), with the fitted power-law model.

In [ ]:
h_c = pipeline.H_C_UNIVERSAL
palette = sns.color_palette("tab10", n_colors=len(trials))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
ax_drag, ax_rheo = axes

# ── Left: Drag vs. Height ───────────────────────────────────────────────────
gamma_pts, mu_pts = [], []

for i, trial in enumerate(trials):
    h = trial["height_mm"]
    rpm = trial["rpm"]
    drag = trial["torque_pct"] / rpm
    color = palette[i]
    label = f"{rpm:g} RPM"

    ax_drag.scatter(h, drag, s=22, alpha=0.75, color=color, label=label)

    def model_fixed_hc(x, A, B, hc=h_c):
        return A / (x + hc) + B

    try:
        popt, _ = curve_fit(
            model_fixed_hc, h, drag,
            p0=[max((drag.max() - drag.min()) * (h.min() + h_c), 1e-3),
                float(np.median(drag[-min(5, len(drag)):]))],
            maxfev=20000,
        )
        A_fit, B_fit = popt
        h_line = np.linspace(h.min(), h.max(), 200)
        ax_drag.plot(h_line, model_fixed_hc(h_line, A_fit, B_fit),
                     color=color, lw=2.0, alpha=0.9)

        if A_fit > 0:
            mu_app = float(pipeline.amplitude_to_viscosity(A_fit))
            gamma = float(pipeline.geo.shear_rate(rpm))
            gamma_pts.append(gamma)
            mu_pts.append(mu_app)
    except Exception:
        pass

ax_drag.set_xlabel("Height (mm)")
ax_drag.set_ylabel(r"Drag  $T / \mathrm{RPM}$")
ax_drag.set_title("Raw Drag Profiles & Fractional Fits")
ax_drag.legend(title="Trial", frameon=True)

# ── Right: Rheology curve (log–log) ─────────────────────────────────────────
if gamma_pts:
    gamma_arr = np.array(gamma_pts)
    mu_arr = np.array(mu_pts)
    ax_rheo.scatter(gamma_arr, mu_arr, s=55, color="#2c3e50",
                    edgecolor="white", linewidth=0.6, zorder=3, label="Data")

    n_fit = result.get("n", np.nan)
    if np.isfinite(n_fit) and np.isfinite(k_cP) and len(gamma_arr) >= 1:
        g_line = np.logspace(np.log10(gamma_arr.min()), np.log10(gamma_arr.max()), 200)
        mu_line = k_cP * g_line ** (n_fit - 1.0)
        ax_rheo.plot(g_line, mu_line, color="#e74c3c", lw=2.2,
                     label=rf"$\mu_{{app}} = K\,\dot{{\gamma}}^{{{n_fit - 1:.2f}}}$")

ax_rheo.set_xscale("log")
ax_rheo.set_yscale("log")
ax_rheo.set_xlabel(r"Shear rate  $\dot{\gamma}$  (s$^{-1}$)")
ax_rheo.set_ylabel(r"Apparent viscosity  $\mu_{app}$  (cP)")
ax_rheo.set_title("Rheology Curve & Power-Law Model")
ax_rheo.legend(frameon=True)

fig.suptitle(f"Rheology Dashboard — {data_path.stem}", fontsize=13, y=1.02)
plt.show()